<a href="https://colab.research.google.com/github/elkins/synth-saxs/blob/main/examples/interactive_tutorials/end_to_end_validation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# End-to-End Validation (synth-suite)

This tutorial showcases the full ecosystem integration. We take a single protein structure and simulate observables across four orthogonal biophysical modalities: **NMR, SAXS, AFM, and Cryo-EM**. 

This multi-modal approach is exactly how modern computational structural biology validates predictions against experimental data.

In [ ]:
import sys

if "google.colab" in sys.modules:
    !pip install -q git+https://github.com/elkins-lab/synth-saxs.git synth-nmr synth-afm synth-cryo-em biotite matplotlib py3Dmol
else:
    sys.path.append("../../")

import biotite.structure.io as strucio
import matplotlib.pyplot as plt
import numpy as np

## 1. The Target Structure
We'll download a common test protein, Ubiquitin (1UBQ).

In [ ]:
import biotite.database.rcsb as rcsb

pdb_file = rcsb.fetch("1UBQ", "pdb", ".")
structure = strucio.load_structure(pdb_file)
protein = structure[~structure.hetero]

coords = protein.coord
elements = protein.element
print(f"Loaded {len(coords)} atoms.")

### 1.5 3D Visualization
Let's look at the structure before running our simulations.

In [ ]:
import py3Dmol

viewer = py3Dmol.view(width=400, height=300)
viewer.addModel(open(pdb_file).read(), "pdb")
viewer.setStyle({"cartoon": {"color": "spectrum"}})
viewer.zoomTo()
viewer.show()

## 2. NMR: Predict Chemical Shifts
Using `synth-nmr`, we predict the $C\alpha$ chemical shifts directly from the 3D topology.

In [ ]:
from synth_nmr import predict_chemical_shifts

shifts = predict_chemical_shifts(protein)
print(f"Predicted CA shift for Residue 1: {shifts.get('A', {}).get(1, {}).get('CA', 'N/A')} ppm")

## 3. SAXS: Predict & Validate
Using `synth-saxs`, we generate the $I(q)$ curve. In a real scenario, you would have experimental SAXS data. We will simulate a noisy 'experimental' dataset and mathematically fit our predicted profile to it to validate the structure!

In [ ]:
from synth_saxs import calculate_saxs_profile
from synth_saxs.fitting import fit_profile

# 1. Simulate Theoretical Profile
q_calc, i_calc = calculate_saxs_profile(protein, q_min=0.01, q_max=0.3, n_points=50)

# 2. Create Mock Experimental Data (with noise)
q_exp = q_calc
i_exp_ideal = i_calc * 12.5 + 0.05  # arbitrary scale and offset
noise = 0.03 * i_exp_ideal
i_exp = np.random.normal(i_exp_ideal, noise)
err_exp = noise

# 3. Fit theoretical to experimental
c_fit, k_fit, chi_sq = fit_profile(i_exp, err_exp, i_calc)
i_fit = c_fit * i_calc + k_fit

print(f"SAXS Validation Complete: chi^2 = {chi_sq:.2f}")

## 4. AFM: Simulate Topography
Using `synth-afm`, we sweep a simulated 1.0 Å tip over the grid to capture the height map.

In [ ]:
from synth_afm import AFMSimulator

afm_sim = AFMSimulator(grid_size=(50, 50), pixel_size=0.8, tip_radius=1.0)
height_map = afm_sim.scan(coords)
print("AFM scan complete.")

## 5. Cryo-EM: Synthesize Density Map
Using `synth-cryo-em`, we drop Gaussian blobs at each atomic coordinate to build a 3D volume.

In [ ]:
from synth_cryo_em.core import generate_density_map

# 1.0 Å spacing, 3.0 Å resolution
density_grid, min_pos = generate_density_map(pdb_file, resolution=3.0, grid_spacing=1.0)
density_map = np.array(density_grid, copy=False)
print("Cryo-EM density synthesis complete.")

## 6. Multi-Modal Dashboard
Let's visualize the results of all four simulations.

In [ ]:
import numpy as np

fig, axes = plt.subplots(2, 2, figsize=(12, 10))

# Top-Left: SAXS Fit
axes[0, 0].errorbar(q_exp, i_exp, yerr=err_exp, fmt="ko", markersize=3, alpha=0.5, label="Exp")
axes[0, 0].plot(q_calc, i_fit, "r-", linewidth=2, label="Fit")
axes[0, 0].set_yscale("log")
axes[0, 0].set_title(f"SAXS Fit (chi^2 = {chi_sq:.1f})")
axes[0, 0].set_xlabel("q")
axes[0, 0].legend()

# Top-Right: AFM
im_afm = axes[0, 1].imshow(height_map, cmap="afmhot")
axes[0, 1].set_title("AFM Topography")
fig.colorbar(im_afm, ax=axes[0, 1])

# Bottom-Left: Cryo-EM Slice
mid_z = density_map.shape[0] // 2
im_em = axes[1, 0].imshow(density_map[mid_z, :, :], cmap="magma")
axes[1, 0].set_title("Cryo-EM Slice (Z=20)")
fig.colorbar(im_em, ax=axes[1, 0])

# Bottom-Right: NMR Chemical Shifts
res_ids = sorted(shifts["A"].keys())
ca_values = [shifts["A"][i].get("CA", 0) for i in res_ids]
axes[1, 1].bar(res_ids[:20], ca_values[:20], color="indigo")
axes[1, 1].set_title("NMR CA Shifts (First 20 Res)")
axes[1, 1].set_xlabel("Residue ID")

plt.tight_layout()
plt.show()